# Cohere Reranker — 프로덕션급 클라우드 재순위화

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

- **Cohere Rerank API**의 특징과 Cross-Encoder(로컬) 대비 장단점 이해하기
- `CohereEmbeddings`와 `CohereRerank`를 결합한 검색 파이프라인 구현하기
- API 키와 모델 선택 방법 익히기

## 사전 지식

- 01-Cross-Encoder-Reranker에서 Two-Stage Retrieval 개념
- Cohere API 키 발급 방법 (`COHERE_API_KEY` 환경변수 필요)

---

```mermaid
flowchart LR
    Q[사용자 쿼리]:::input --> CE[Cohere Embeddings<br/>초기 벡터 검색<br/>k=10]:::process
    CE --> CR[Cohere Rerank API<br/>cloud 재정렬<br/>top_n=3]:::external
    CR --> R[관련성 점수 포함<br/>상위 3개 문서]:::output

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef external fill:#d1ecf1,stroke:#17a2b8,color:#0c5460
```

## Cohere vs Cross-Encoder (로컬) 비교

| 특징 | Cohere Reranker | Cross-Encoder (로컬) |
|------|:---:|:---:|
| 정확도 | 최고 수준 | 높음 |
| 다국어 | 100+ 언어 | 모델 의존 |
| 비용 | API 사용료 | 인프라 비용 |
| 설정 | 즉시 사용 | 모델 다운로드 필요 |
| 추천 용도 | 프로덕션 서비스 | 개발·테스트 |

> **실무 팁**: `COHERE_API_KEY`가 없어도 노트북의 핵심 개념을 확인할 수 있어요. API 키 없이도 코드 흐름을 이해하고 실습해보세요.

> 🎯 **강의 포인트**: Cohere Reranker는 인프라 관리 없이 바로 프로덕션 수준의 재정렬을 제공합니다. 한국어를 포함한 100개 이상 언어를 지원하므로, 한국어 RAG 서비스를 출시할 때 가장 빠른 선택지입니다.

> 🔑 **핵심 개념**: 클라우드 Reranker(Cohere, Jina)는 API 호출당 비용이 발생합니다. 실시간 서비스라면 레이턴시와 비용을 함께 고려해야 해요. 배치 처리가 가능하다면 캐싱 전략도 검토하세요.

## 1. 환경 설정

### 1.1 Cohere API 키 발급

1. [Cohere 웹사이트](https://dashboard.cohere.com/api-keys)에서 API 키 발급
2. `.env` 파일에 API 키 추가:

```bash
COHERE_API_KEY="your-cohere-api-key"
```

### 1.2 Cohere 모델 정보

**Embedding 모델**:
- `embed-multilingual-v3.0`: 최신 다국어 임베딩 (권장)
- `embed-multilingual-light-v3.0`: 경량 버전
- `embed-multilingual-v2.0`: 이전 버전

**Reranker 모델**:
- `rerank-multilingual-v3.0`: 최신 다국어 재정렬 (권장)
- `rerank-english-v3.0`: 영어 특화 모델
- `rerank-multilingual-v2.0`: 이전 버전


In [1]:
# 필요한 라이브러리 import
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
from dotenv import load_dotenv
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_cohere import CohereEmbeddings, CohereRerank
from langchain.retrievers import ContextualCompressionRetriever

# 환경 변수 로드
load_dotenv()
COHERE_API_KEY = os.getenv("COHERE_API_KEY")
if not COHERE_API_KEY:
    print("⚠️ COHERE_API_KEY가 없어 Cohere 예제를 실행할 수 없습니다.")

## 2. 문서 출력 도우미 함수


In [2]:
def pretty_print_docs(docs, show_scores=True):
    """검색된 문서를 보기 좋게 출력하는 함수"""
    print(f"\n{'=' * 100}")
    print(f"총 {len(docs)}개 문서 검색됨")
    print(f"{'=' * 100}\n")
    
    for i, doc in enumerate(docs, 1):
        print(f"[문서 {i}]")
        
        # Cohere Reranker는 relevance_score를 metadata에 추가
        if show_scores and 'relevance_score' in doc.metadata:
            score = doc.metadata['relevance_score']
            print(f"관련성 점수: {score:.4f}")
        
        print(f"내용: {doc.page_content}")
        print(f"{'─' * 100}\n")


## 3. Cohere Embeddings로 벡터 검색 시스템 구축

Cohere의 다국어 임베딩 모델을 사용하여 초기 검색 시스템을 구축합니다.


In [3]:
# 1단계: 문서 로드
documents = TextLoader("./data/appendix-keywords.txt").load()

print(f"✅ 문서 로드 완료")
print(f"   - 문서 수: {len(documents)}")
print(f"   - 총 길이: {len(documents[0].page_content)} 문자")


✅ 문서 로드 완료
   - 문서 수: 1
   - 총 길이: 5733 문자


In [4]:
# 2단계: 텍스트 분할
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

texts = text_splitter.split_documents(documents)

print(f"✅ 문서 분할 완료")
print(f"   - 청크 수: {len(texts)}")
print(f"   - 평균 청크 길이: {sum(len(t.page_content) for t in texts) // len(texts)} 문자")


✅ 문서 분할 완료
   - 청크 수: 15
   - 평균 청크 길이: 390 문자


In [5]:
# ---------------------------------------------------
# Cohere 임베딩으로 FAISS 벡터스토어 생성
# ---------------------------------------------------
# ============================================================
# TODO: CohereEmbeddings와 FAISS로 벡터스토어를 만들고 검색기를 설정하세요
# 힌트: CohereEmbeddings(model="embed-multilingual-v3.0"), k=10
# 예상 결과: "Cohere 벡터스토어 생성 완료" 출력
# ============================================================

# 3단계: Cohere 임베딩 모델 초기화
if not COHERE_API_KEY:
    embeddings = None
    vectorstore = None
    base_retriever = None
    print("⚠️ COHERE_API_KEY가 없어 Cohere 임베딩/벡터스토어 예제를 건너뜁니다.")
else:
    # embed-multilingual-v3.0: Cohere의 최신 다국어 임베딩 모델 (한국어 포함)
    embeddings = CohereEmbeddings(
        model="embed-multilingual-v3.0",
        cohere_api_key=COHERE_API_KEY,
    )

    # 4단계: FAISS 벡터스토어 생성 및 검색기 설정
    vectorstore = FAISS.from_documents(texts, embeddings)
    base_retriever = vectorstore.as_retriever(
        search_kwargs={"k": 10}  # 초기 검색: 10개 문서
    )

    print("✅ Cohere 벡터스토어 생성 완료")
    print(f"   - 임베딩 모델: embed-multilingual-v3.0")
    print(f"   - 벡터스토어: FAISS")
    print(f"   - 초기 검색 수: 10개")

✅ Cohere 벡터스토어 생성 완료
   - 임베딩 모델: embed-multilingual-v3.0
   - 벡터스토어: FAISS
   - 초기 검색 수: 10개


In [6]:
# ---------------------------------------------------
# Reranker 적용 전 기본 검색 결과 확인
# ---------------------------------------------------
# ============================================================
# TODO: base_retriever로 쿼리를 검색하고 결과를 출력하세요
# 힌트: invoke(query) 호출, show_scores=False로 점수 숨기기
# 예상 결과: 10개 문서가 Cohere 임베딩 유사도 순으로 출력됨
# ============================================================

# 기본 검색 테스트
query = "딥러닝 모델이 문맥을 이해하는 메커니즘은?"

if base_retriever is None:
    print("⚠️ Cohere 벡터스토어가 없어 기본 검색 단계를 건너뜁니다.")
else:
    print(f"\n🔍 검색 쿼리: {query}\n")

    # Reranker 적용 전 기본 검색
    docs_before = base_retriever.invoke(query)

    print("📊 [Reranker 적용 전] Cohere 임베딩 기반 검색 결과:")
    pretty_print_docs(docs_before, show_scores=False)


🔍 검색 쿼리: 딥러닝 모델이 문맥을 이해하는 메커니즘은?

📊 [Reranker 적용 전] Cohere 임베딩 기반 검색 결과:

총 10개 문서 검색됨

[문서 1]
내용: Deep Learning

정의: 딥러닝은 인공신경망을 이용하여 복잡한 문제를 해결하는 머신러닝의 한 분야입니다. 이는 데이터에서 고수준의 표현을 학습하는 데 중점을 둡니다.
예시: 이미지 인식, 음성 인식, 자연어 처리 등에서 딥러닝 모델이 활용됩니다.
연관키워드: 인공신경망, 머신러닝, 데이터 분석

Schema

정의: 스키마는 데이터베이스나 파일의 구조를 정의하는 것으로, 데이터가 어떻게 저장되고 조직되는지에 대한 청사진을 제공합니다.
예시: 관계형 데이터베이스의 테이블 스키마는 열 이름, 데이터 타입, 키 제약 조건 등을 정의합니다.
연관키워드: 데이터베이스, 데이터 모델링, 데이터 관리

DataFrame
────────────────────────────────────────────────────────────────────────────────────────────────────

[문서 2]
내용: DataFrame

정의: DataFrame은 행과 열로 이루어진 테이블 형태의 데이터 구조로, 주로 데이터 분석 및 처리에 사용됩니다.
예시: 판다스 라이브러리에서 DataFrame은 다양한 데이터 타입의 열을 가질 수 있으며, 데이터 조작과 분석을 용이하게 합니다.
연관키워드: 데이터 분석, 판다스, 데이터 처리

Attention 메커니즘

정의: Attention 메커니즘은 딥러닝에서 중요한 정보에 더 많은 '주의'를 기울이도록 하는 기법입니다. 이는 주로 시퀀스 데이터(예: 텍스트, 시계열 데이터)에서 사용됩니다.
예시: 번역 모델에서 Attention 메커니즘은 입력 문장의 중요한 부분에 더 집중하여 정확한 번역을 생성합니다.
연관키워드: 딥러닝, 자연어 처리, 시퀀스 모델링

판다스 (Pandas)
───────────────────────────────────

## 4. Cohere Reranker 적용

Cohere의 Rerank API를 사용하여 검색 결과를 재정렬합니다.


In [7]:
# ---------------------------------------------------
# Cohere Rerank API로 압축 검색기 구성
# ---------------------------------------------------
# ============================================================
# TODO: CohereRerank와 ContextualCompressionRetriever로 재정렬 파이프라인을 만드세요
# 힌트: CohereRerank(model="rerank-multilingual-v3.0", top_n=3)
# 예상 결과: "Cohere Reranker 설정 완료" 출력
# ============================================================

if base_retriever is None:
    compression_retriever = None
    print("⚠️ Cohere 벡터스토어가 없어 Reranker 단계를 건너뜁니다.")
else:
    # 1단계: Cohere Reranker 초기화
    # rerank-multilingual-v3.0: 최신 다국어 재정렬 모델 (100+ 언어 지원)
    compressor = CohereRerank(
        model="rerank-multilingual-v3.0",
        top_n=3,  # 10개 후보에서 상위 3개만 최종 반환
        cohere_api_key=COHERE_API_KEY,
    )

    # 2단계: 압축 검색기 생성
    compression_retriever = ContextualCompressionRetriever(
        base_compressor=compressor,
        base_retriever=base_retriever
    )

    print("✅ Cohere Reranker 설정 완료")
    print(f"   - 모델: rerank-multilingual-v3.0")
    print(f"   - 초기 검색: 10개 → 최종 반환: 3개")
    print(f"   - 지원 언어: 100+ (한국어 포함)")

✅ Cohere Reranker 설정 완료
   - 모델: rerank-multilingual-v3.0
   - 초기 검색: 10개 → 최종 반환: 3개
   - 지원 언어: 100+ (한국어 포함)


In [8]:
# ---------------------------------------------------
# Reranker 적용 후 검색 결과 확인
# ---------------------------------------------------
# ============================================================
# TODO: compression_retriever로 같은 쿼리를 검색하고 관련성 점수를 확인하세요
# 힌트: invoke(query) 후 pretty_print_docs(docs, show_scores=True)
# 예상 결과: 3개 문서와 각각의 Cohere 관련성 점수 출력
# ============================================================

if compression_retriever is None:
    docs_after = []
    print("⚠️ Cohere Reranker가 설정되지 않아 재정렬 단계를 건너뜁니다.")
else:
    # Reranker 적용 검색
    print(f"🔍 검색 쿼리: {query}\n")

    docs_after = compression_retriever.invoke(query)

    print("🎯 [Reranker 적용 후] Cohere Rerank 결과:")
    pretty_print_docs(docs_after, show_scores=True)

🔍 검색 쿼리: 딥러닝 모델이 문맥을 이해하는 메커니즘은?

🎯 [Reranker 적용 후] Cohere Rerank 결과:

총 3개 문서 검색됨

[문서 1]
관련성 점수: 0.9097
내용: DataFrame

정의: DataFrame은 행과 열로 이루어진 테이블 형태의 데이터 구조로, 주로 데이터 분석 및 처리에 사용됩니다.
예시: 판다스 라이브러리에서 DataFrame은 다양한 데이터 타입의 열을 가질 수 있으며, 데이터 조작과 분석을 용이하게 합니다.
연관키워드: 데이터 분석, 판다스, 데이터 처리

Attention 메커니즘

정의: Attention 메커니즘은 딥러닝에서 중요한 정보에 더 많은 '주의'를 기울이도록 하는 기법입니다. 이는 주로 시퀀스 데이터(예: 텍스트, 시계열 데이터)에서 사용됩니다.
예시: 번역 모델에서 Attention 메커니즘은 입력 문장의 중요한 부분에 더 집중하여 정확한 번역을 생성합니다.
연관키워드: 딥러닝, 자연어 처리, 시퀀스 모델링

판다스 (Pandas)
────────────────────────────────────────────────────────────────────────────────────────────────────

[문서 2]
관련성 점수: 0.1497
내용: JSON

정의: JSON(JavaScript Object Notation)은 경량의 데이터 교환 형식으로, 사람과 기계 모두에게 읽기 쉬운 텍스트를 사용하여 데이터 객체를 표현합니다.
예시: {"이름": "홍길동", "나이": 30, "직업": "개발자"}는 JSON 형식의 데이터입니다.
연관키워드: 데이터 교환, 웹 개발, API

Transformer

정의: 트랜스포머는 자연어 처리에서 사용되는 딥러닝 모델의 한 유형으로, 주로 번역, 요약, 텍스트 생성 등에 사용됩니다. 이는 Attention 메커니즘을 기반으로 합니다.
예시: 구글 번역기는 트랜스포머 모델을 사용하여 다양한 언어 간의 번역을 수행합니다.
연관키워드: 딥러닝,

## 5. 결과 비교 및 분석


In [9]:
if base_retriever is None or compression_retriever is None:
    print("⚠️ Cohere API 키가 없어 효과 분석을 건너뜁니다.")
else:
    print("\n" + "=" * 100)
    print("📊 Cohere Reranker 효과 분석")
    print("=" * 100)

    print(f"\n[검색 쿼리]")
    print(f"  {query}")

    print(f"\n[Reranker 적용 전] - 상위 3개")
    for i, doc in enumerate(docs_before[:3], 1):
        preview = doc.page_content.replace('\n', ' ')[:100]
        print(f"  {i}. {preview}...")

    print(f"\n[Reranker 적용 후] - 상위 3개 + 관련성 점수")
    for i, doc in enumerate(docs_after, 1):
        preview = doc.page_content.replace('\n', ' ')[:100]
        score = doc.metadata.get('relevance_score', 0)
        print(f"  {i}. [점수: {score:.4f}] {preview}...")

    print("\n💡 Cohere Reranker의 장점:")
    print("  ✅ 높은 정확도: State-of-the-art 재정렬 성능")
    print("  ✅ 다국어 지원: 한국어 문서에 최적화")
    print("  ✅ 관리형 서비스: 인프라 관리 불필요")
    print("  ✅ 프로덕션 준비: 안정적인 API 서비스")



📊 Cohere Reranker 효과 분석

[검색 쿼리]
  딥러닝 모델이 문맥을 이해하는 메커니즘은?

[Reranker 적용 전] - 상위 3개
  1. Deep Learning  정의: 딥러닝은 인공신경망을 이용하여 복잡한 문제를 해결하는 머신러닝의 한 분야입니다. 이는 데이터에서 고수준의 표현을 학습하는 데 중점을 둡니다. 예시...
  2. DataFrame  정의: DataFrame은 행과 열로 이루어진 테이블 형태의 데이터 구조로, 주로 데이터 분석 및 처리에 사용됩니다. 예시: 판다스 라이브러리에서 DataFra...
  3. 정의: LLM은 대규모의 텍스트 데이터로 훈련된 큰 규모의 언어 모델을 의미합니다. 이러한 모델은 다양한 자연어 이해 및 생성 작업에 사용됩니다. 예시: OpenAI의 GPT 시리...

[Reranker 적용 후] - 상위 3개 + 관련성 점수
  1. [점수: 0.9097] DataFrame  정의: DataFrame은 행과 열로 이루어진 테이블 형태의 데이터 구조로, 주로 데이터 분석 및 처리에 사용됩니다. 예시: 판다스 라이브러리에서 DataFra...
  2. [점수: 0.1497] JSON  정의: JSON(JavaScript Object Notation)은 경량의 데이터 교환 형식으로, 사람과 기계 모두에게 읽기 쉬운 텍스트를 사용하여 데이터 객체를 표현합...
  3. [점수: 0.1122] Deep Learning  정의: 딥러닝은 인공신경망을 이용하여 복잡한 문제를 해결하는 머신러닝의 한 분야입니다. 이는 데이터에서 고수준의 표현을 학습하는 데 중점을 둡니다. 예시...

💡 Cohere Reranker의 장점:
  ✅ 높은 정확도: State-of-the-art 재정렬 성능
  ✅ 다국어 지원: 한국어 문서에 최적화
  ✅ 관리형 서비스: 인프라 관리 불필요
  ✅ 프로덕션 준비: 안정적인 API 서비스


## 6. 다양한 쿼리 테스트


In [10]:
# ---------------------------------------------------
# 다양한 쿼리로 Cohere Reranker 성능 검증
# ---------------------------------------------------
# ============================================================
# TODO: 여러 쿼리를 순회하며 Cohere Reranker로 검색하고 관련성 점수를 비교하세요
# 힌트: for 루프로 test_queries 순회, compression_retriever.invoke(test_query)
# 예상 결과: 각 쿼리별 상위 3개 문서와 Cohere 관련성 점수 출력
# ============================================================

if compression_retriever is None:
    print("⚠️ Cohere Reranker가 없어 추가 테스트를 건너뜁니다.")
else:
    # 다양한 쿼리로 Cohere Reranker 성능 검증
    test_queries = [
        "파이썬에서 데이터를 표 형식으로 다루는 라이브러리는?",
        "데이터베이스에서 특정 조건의 데이터를 가져오는 언어는?",
        "텍스트를 작은 단위로 나누는 것을 무엇이라고 하나요?"
    ]

    print("\n" + "=" * 100)
    print("🧪 Cohere Reranker 다양한 쿼리 테스트")
    print("=" * 100)

    for idx, test_query in enumerate(test_queries, 1):
        print(f"\n{'─' * 100}")
        print(f"[테스트 {idx}] 쿼리: {test_query}")
        print(f"{'─' * 100}")
        
        # Cohere Reranker 적용
        reranked_docs = compression_retriever.invoke(test_query)
        
        print(f"\n✅ Cohere가 선택한 상위 {len(reranked_docs)}개 문서:")
        
        for i, doc in enumerate(reranked_docs, 1):
            preview = doc.page_content.replace('\n', ' ')[:80]
            score = doc.metadata.get('relevance_score', 0)
            print(f"\n  [{i}] 점수: {score:.4f}")
            print(f"      내용: {preview}...")


🧪 Cohere Reranker 다양한 쿼리 테스트

────────────────────────────────────────────────────────────────────────────────────────────────────
[테스트 1] 쿼리: 파이썬에서 데이터를 표 형식으로 다루는 라이브러리는?
────────────────────────────────────────────────────────────────────────────────────────────────────

✅ Cohere가 선택한 상위 3개 문서:

  [1] 점수: 0.7699
      내용: 판다스 (Pandas)  정의: 판다스는 파이썬 프로그래밍 언어를 위한 데이터 분석 및 조작 도구를 제공하는 라이브러리입니다. 이는 데이터 분석...

  [2] 점수: 0.0765
      내용: DataFrame  정의: DataFrame은 행과 열로 이루어진 테이블 형태의 데이터 구조로, 주로 데이터 분석 및 처리에 사용됩니다. 예시:...

  [3] 점수: 0.0615
      내용: SQL  정의: SQL(Structured Query Language)은 데이터베이스에서 데이터를 관리하기 위한 프로그래밍 언어입니다. 데이터 ...

────────────────────────────────────────────────────────────────────────────────────────────────────
[테스트 2] 쿼리: 데이터베이스에서 특정 조건의 데이터를 가져오는 언어는?
────────────────────────────────────────────────────────────────────────────────────────────────────

✅ Cohere가 선택한 상위 3개 문서:

  [1] 점수: 0.2978
      내용: SQL  정의: SQL(Structured Query Language)은 데이터베이스에서 데이터를 관리하기 위한 프로

## 7. 핵심 정리

### Reranker 비교

| 특징 | Cohere Reranker | Cross-Encoder (로컬) |
|------|:---:|:---:|
| 정확도 | 최고 수준 | 높음 |
| 다국어 | 100+ 언어 | 모델 의존 |
| 비용 | API 사용료 | 무료 (인프라 비용) |
| 관리 | 완전 관리형 | 직접 관리 |
| 추천 용도 | 프로덕션 서비스 | 개발/테스트 |

### 기본 사용법

```python
from langchain_cohere import CohereRerank

compressor = CohereRerank(
    model="rerank-multilingual-v3.0",
    top_n=3,
)
```

### 주요 파라미터

| 파라미터 | 설명 | 기본값 |
|---------|------|--------|
| `model` | Reranker 모델명 | `rerank-multilingual-v3.0` |
| `top_n` | 최종 반환 문서 수 | 3 |
| `cohere_api_key` | Cohere API 키 | 환경변수 `COHERE_API_KEY` |

> 💡 **실무 팁**: Cohere는 무료 체험 크레딧을 제공합니다. 먼저 무료 API 키로 테스트하고, 실제 트래픽이 발생할 때 유료 플랜으로 전환하는 것을 권장해요.

> ⚠️ **자주 하는 실수**: `embed-multilingual-v3.0` 임베딩 모델과 `rerank-multilingual-v3.0` 리랭커 모델은 별개입니다. 임베딩 생성에 쓴 모델과 동일한 제공사의 리랭커를 쓸 필요는 없어요. FAISS + Cohere Rerank 조합도 완전히 유효합니다.

## 마무리 정리

이 노트북에서 배운 내용을 정리해요:

### Reranker 선택 기준

| 상황 | 추천 |
|------|------|
| 개발·테스트 환경 | Cross-Encoder (로컬, 무료) |
| 한국어 프로덕션 서비스 | **Cohere** (다국어, 관리형) |
| 긴 문서(2K+ 토큰) | Jina Reranker (8K 지원) |
| 완전한 데이터 통제 | Cross-Encoder (온프레미스) |

### 다음 노트북 예고

**03-Jina-Reranker**: 최대 8K 토큰까지 처리 가능한 Jina Reranker를 배워요. 긴 문서 재순위화에서 차별화된 성능을 확인해볼 수 있어요.